Tento výukový materiál vychází z https://github.com/KeithGalli/pandas/tree/master
Video k tomuto výukovému materiálu je zde: https://youtu.be/vmEHCJofslg


# Stažení dat

- Soubor `pokemon_data.csv` se nachází ve stejné složce jako tento notebook, tedy v `/home/teaching/NOFY056/plesv6am`. Odtud si jej můžete zkopírovat.

- Další možností je stáhnout jej z https://raw.githubusercontent.com/KeithGalli/pandas/master/pokemon_data.csv. Pokud máte nainstalovaný modul `requests`, odkomentujte a spusťte následující buňku. Upozorňujeme, že `requests` není součástí jádra `praktikum`.

- Soubor si můžete stáhnout i jiným způsobem, například pomocí webového prohlížeče.

In [ ]:
# import requests

# def download_file(url, filename):
#     try:
#         response = requests.get(url, stream=True)
#         response.raise_for_status()  # Raise an exception for bad status codes

#         with open(filename, 'wb') as file:
#             for chunk in response.iter_content(chunk_size=8192):
#                 file.write(chunk)
#         print(f"File '{filename}' downloaded successfully.")

#     except requests.exceptions.RequestException as e:
#         print(f"Error downloading file: {e}")

# # Example usage:
# file_url = "https://raw.githubusercontent.com/KeithGalli/pandas/master/pokemon_data.csv" # Note: using raw content link
# file_name = "pokemon_data.csv"

# download_file(file_url, file_name)

# Načtení dat do pandas

- Soubor je jednoduchý textový soubor CSV (hodnoty oddělené čárkami). Do objektu DataFrame knihovny pandas jej můžete načíst funkcí `pd.read_csv()`.

In [ ]:
# Open the downloaded file with pandas.

import pandas as pd

# Assuming the file is named 'pokemon_data.csv' in the current directory
df = pd.read_csv('pokemon_data.csv')#, index_col = 'Name')
print(df.head()) # Print the first few rows of the DataFrame

In [ ]:
# The dataframe is a class defined in pandas.
print(type(df))

## Přístup k datům

Načtěte názvy sloupců:

In [ ]:
df.columns

Získejte sloupec jako objekt `pandas.core.series.Series`:

In [ ]:
print(df['Name'])

Získejte DataFrame obsahující pouze vybrané sloupce:

In [ ]:
small_df = df[['Name', 'Type 1', 'Type 2', 'HP']]
small_df # Jupyter notebooks display dataframes nicer without the `print` function.
df.loc[df['Name'] == 'Ivysaur']

Protože jsme ve funkci `read_csv` výše zvolili `index_col='Name'`, můžeme jako index použít název pokémona. Všimněte si použití metody `loc` pro výběr sloupců a řádků. Tato metoda vybírá podle předem daných popisků řádků a sloupců. V tomto notebooku ji budeme používat často!

In [ ]:
df.loc['Ivysaur']

Vráceným objektem je `pandas.Series`. K jeho prvkům můžeme přistupovat pomocí hranatých závorek, například `pokemon['Type 2']`:

In [ ]:
df.loc['Ivysaur']['Type 2']

Metoda `iloc` vybírá sloupce a řádky podle jejich pozice v DataFrame. Pozice se číslují celými čísly. Následující kód vrátí třetí řádek našeho DataFrame:

In [ ]:
df.iloc[2]

Výběr výřezu řádků pomocí `iloc`:

In [ ]:
print(df.iloc[0:4])

Načtěte „buňku“ na souřadnicích [2,1] (řádek, sloupec):

In [ ]:
print(df.iloc[2,1])

Vyberte podmnožinu řádků (pouze ty, kde má pokémon `Type 1` hodnotu `Grass`).

In [ ]:
df.loc[df['Type 1'] == "Grass"]

Všimněte si, že výraz ```df['Type 1'] == "Grass"``` vytvoří pole logických hodnot. Metoda `loc` tedy přijímá jako vstup pole logických hodnot. S logickými sloupci proto můžeme provádět i následující operace:

In [ ]:
df.loc[df['Legendary']]

Použijte více výběrových podmínek pomocí logického operátoru AND `&`. Všimněte si závorek kolem podmínky `df['Type 1'] == "Fire"`. Musí tam být, protože operátor `&` (bitový AND) má nižší prioritu než operátor rovnosti `==`.

In [ ]:
df.loc[df['Legendary'] & (df['Type 1'] == "Fire")]

Logická operace OR:

In [ ]:
df.loc[(df['Type 1'] == 'Grass') | (df['Type 1'] == "Fire")]

Po filtrování se můžeme například zeptat, jaká je nejvyšší hodnota „Attack“ u vybraných pokémonů.

In [ ]:
fire_legendary_pokemon = df.loc[(df['Type 1'] == 'Fire') & (df['Legendary'] == True)]

if not fire_legendary_pokemon.empty:
  pokemon_with_highest_attack = fire_legendary_pokemon.loc[fire_legendary_pokemon['Attack'].idxmax()]
  print(pokemon_with_highest_attack)

Operátor `~` představuje logickou negaci NOT. Následující výraz vybere všechny pokémony, kteří nejsou legendární a jejichž hodnota „HP“ je vyšší než 150.

In [ ]:
df.loc[~df['Legendary'] & (df['HP'] > 150)]

Počet druhů v „Type 1“ zjistíte pomocí metody `value_counts`.

In [ ]:
df['Type 1'].value_counts()

Vyberte pokémony s hodnotou Defense v intervalu `(40, 60)`:

In [ ]:
new_df = df.loc[df['Defense'].between(40, 60)]
new_df.head()

Pokud chceme pokémony s určitými hodnotami „Defense“, použijeme metodu `isin`.

In [ ]:
df.loc[df['Defense'].isin([40, 70, 30, 60])].head()

Vyberte 5 % pokémonů s nejvyšší rychlostí. K tomu se hodí metoda `quantile`! (Nevybereme přesně 5 % pokémonů, protože více pokémonů mívá stejnou rychlost.)

In [ ]:
new_df = df.loc[df['Speed'] > df['Speed'].quantile(0.95)]
print(new_df.shape)
print(df.shape)
print(new_df.shape[0] / df.shape[0])

## Řazení a popis dat

Řádky podle hodnoty „HP“ sestupně seřadíte takto:

In [ ]:
sorted_df = df.sort_values('HP', ascending = False)
print(sorted_df.head())

Řádky lze řadit také podle několika vlastností. Nejprve se seřadí sestupně podle hodnoty „HP“. Pokud má více pokémonů stejnou hodnotu „HP“, seřadí se vzestupně podle hodnoty „Attack“.

In [ ]:
df.sort_values(['HP', 'Attack'], ascending = [True, False])

## Úpravy dat

Vytvořte nový sloupec obsahující součet hodnot ze 6 vybraných sloupců:

In [ ]:
df['Total'] = df['HP'] + df['Attack'] + df['Defense'] + df['Sp. Atk'] + df['Sp. Def'] + df['Speed']
print(df['Total'])

Odstraňte sloupec

In [ ]:
df = df.drop(columns=['Total'])

Vytvořte jej znovu a vypočítejte stejný součet jako předtím:

In [ ]:
df['Total'] = df.iloc[:, 4:10].sum(axis = 1)
print(df['Total'])

Změňte pořadí sloupců:

In [ ]:
cols = list(df.columns)
df = df[cols[0:4] + [cols[-1]] + cols[4:11]]

df.head(5)

Metoda `reset_index` přenastaví indexy podle zvoleného způsobu. `inplace=True` znamená, že se nevytvoří nový DataFrame. `drop=True` znamená, že se předchozí index z DataFrame odstraní. Při `drop=False` se starý index přidá do DataFrame jako nový sloupec.

In [ ]:
new_df = df.loc[(df['Type 1'] == 'Grass') & (df['Type 2'] == 'Poison') & (df['HP'] > 70)]
new_df.reset_index(drop = True, inplace = True)

Podmíněné úpravy

In [ ]:
df.loc[df['Attack'] > 100, ['Generation','Legendary']] = [10, True]
df.loc[df['Attack'] > 100]

# Vytváření DataFrame

- Příklady vytvoření nového DataFrame:

In [ ]:
# Method 1: From a dictionary
data = {'col1': [1, 2, 3], 'col2': [4, 5, 6]}
df_from_dict = pd.DataFrame(data)
df_from_dict


In [ ]:
# Method 2: From a list of lists
data = [[1, 4], [2, 5], [3, 6]]
columns = ['col1', 'col2']
df_from_list = pd.DataFrame(data, columns = columns)
df_from_list

In [ ]:
# Method 3: From a NumPy array
import numpy as np
data = np.array([[1, 4], [2, 5], [3, 6]])
columns = ['col1', 'col2']
df_from_array = pd.DataFrame(data, columns=columns)
df_from_array

In [ ]:
# Method 4: From another DataFrame (copying or selecting specific columns/rows)
df_copy = df.copy() #creates a copy of the original dataframe named df
df_copy.head()
df_subset = df[['Type 1', 'HP']] # Select specific columns into new dataframe
df_subset.head()

In [ ]:
# Method 5:  Empty DataFrame with specified column names
df_empty = pd.DataFrame(columns=['A', 'B', 'C'])
df_empty

# Spojování DataFrame

- Příklady použití metody concat:

In [ ]:
import pandas as pd

# Sample DataFrames (replace with your actual DataFrames)
data1 = {'col1': [1, 2, 3], 'col2': [4, 5, 6]}
df1 = pd.DataFrame(data1)

data2 = {'col1': [7, 8, 9], 'col2': [10, 11, 12]}
df2 = pd.DataFrame(data2)

# Concatenate DataFrames along rows (axis = 0, default)
df_concat_rows = pd.concat([df1, df2])
df_concat_rows

In [ ]:
# Ignoring index
df_ignore_index = pd.concat([df1, df2], ignore_index=True)
print("\nConcatenated along rows with ignored index:\n", df_ignore_index)

# Concatenate DataFrames along columns (axis=1)
df_concat_cols = pd.concat([df1, df2], axis=1)
print("\nConcatenated along columns:\n", df_concat_cols)

# Concatenate with keys to create a hierarchical index
df_concat_keys = pd.concat([df1, df2], keys=['df1', 'df2'])
print("\nConcatenated with keys:\n", df_concat_keys)

# Accessing data with hierarchical index
print("\nAccessing data from hierarchical index:")
print(df_concat_keys.loc['df1']) # Accessing data in df1

# Handling different column names (outer join)
data3 = {'col3': [13, 14, 15], 'col4': [16, 17, 18]}
df3 = pd.DataFrame(data3)

df_concat_outer = pd.concat([df1, df3], axis=1)
print("\nConcatenated with different column names (outer join):\n", df_concat_outer)

# Handling different column names (inner join)
df_concat_inner = pd.concat([df1, df3], axis=1, join="inner") # Only common columns will be kept
print("\nConcatenated with different column names (inner join):\n", df_concat_inner) # Should show nothing as no common columns


## Uložení dat (export do požadovaného formátu)

In [ ]:
# df.to_csv('modified.csv', index=False)

#df.to_excel('modified.xlsx', index=False)

df.to_csv('modified.txt', index=False, sep='\t')




## Vykreslování grafů

In [ ]:
# prompt: Draw the histogram of the pokemons' HP values. Use dataframe.plot method for this

import matplotlib.pyplot as plt

df['HP'].plot(kind='hist', bins=20)  # Adjust the number of bins as needed
plt.xlabel('HP Value')
plt.ylabel('Frequency')
plt.title('Histogram of Pokemon HP Values')
plt.show()

In [ ]:
df[df['Type 1'] == 'Grass']  .plot(kind='scatter', x='Attack', y='Defense', color='red',  label='Grass')
plt.xlabel('Attack')
plt.ylabel('Defense')
plt.title('Attack vs. Defense')
plt.show()

In [ ]:
# prompt: Draw scatter plot of the Attack vs. Defense values. Use built-in pandas methods for this. Plot it for Grass and Psychic type 1 pokemons to one single figure. Use pandas methods for this!

# Filter for Grass and Psychic type 1 Pokémon
grass_pokemon = df[df['Type 1'] == 'Grass']
psychic_pokemon = df[df['Type 1'] == 'Psychic']

plt.scatter(grass_pokemon['Attack'], grass_pokemon['Defense'], label='Grass', alpha=0.7, color='green')
plt.scatter(psychic_pokemon['Attack'], psychic_pokemon['Defense'], label='Psychic', alpha=0.7, color='violet')

plt.xlabel('Attack')
plt.ylabel('Defense')
plt.legend()
plt.show()

In [ ]:
# prompt: Plot bar chart comparing the HP value of Grass and Fire type 1 pokemons. Make 20 bins in HP and normalize the histograms before plotting


plt.hist([grass_pokemon['HP'], psychic_pokemon['HP']], bins=10, label=['Grass', 'Psychic'], color=['green', 'violet'], density=True)
plt.xlabel('HP Value')
plt.ylabel('Normalized frequency')
plt.legend()
plt.show()

In [ ]:
# prompt: draw frequencies of the different pokemon type 1 values

# Count the frequencies of each Type 1
type1_counts = df['Type 1'].value_counts()

# Create the bar plot
plt.figure(figsize=(10, 10))  # Adjust figure size for better readability
type1_counts.plot(kind='pie')
plt.show()

In [ ]:

# Count the frequencies of each Type 1
type1_counts = df['Type 1'].value_counts()

# Create the bar plot
type1_counts.plot(kind='bar')
plt.xlabel("Pokemon Type 1")
plt.ylabel("Frequency")
plt.xticks(rotation=45, ha='right')  # Rotate x-axis labels for better visibility
plt.tight_layout() # Adjust layout to prevent labels from overlapping
plt.show()